In [1]:
from urllib.parse import urlencode, quote

def build_naukri_url(keyword: str, location: str, experience: int, job_age: int = 15) -> str:
    # Format the slug portion so location comes right after "jobs-in"
    formatted_keyword = keyword.strip().lower().replace(" ", "-")
    formatted_location = location.strip().lower().replace(" ", "-")
    
    base_path = f"https://www.naukri.com/{formatted_keyword}-jobs-in-{formatted_location}"
    
    # Keep only experience and jobAge as query parameters
    params = {
        "experience": experience,
        "jobAge": job_age
    }
    
    query_string = urlencode(params, quote_via=quote)
    
    return f"{base_path}?{query_string}"

In [2]:
url = build_naukri_url('AI ML Engineer deloitte accenture', 'Hyderabad', 5, 15)


In [12]:
import asyncio
from playwright.async_api import async_playwright
from playwright_stealth import Stealth

async def fetch_page_on_load(url: str) -> str:
    async with Stealth().use_async(async_playwright()) as p:
        browser = await p.chromium.launch(
            headless=False,
            args=[
                "--disable-blink-features=AutomationControlled",
                "--disable-notifications",           # Disables Web and System Notifications
                "--disable-desktop-notifications",   # Prevents D-Bus notification integration
                "--no-default-browser-check"
            ]
        )
        
        context = await browser.new_context(
            viewport={"width": 120, "height": 180},
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
        )
        
        page = await context.new_page()

        try:
            print(f"Navigating to {url}...")
            
            # 1. Option A: Pass wait_until="load" to goto
            await page.goto(url, wait_until="load", timeout=60000)
            
            # 2. Option B (Alternative): Explicitly wait for the 'load' state
            await page.wait_for_load_state("load")
            
            # Optional: 6 second pause after load event fires
            await page.wait_for_timeout(6000)

            # Get final page HTML
            html_content = await page.content()
            return html_content

        finally:
            await context.close()
            await browser.close()

# In Jupyter Notebook cell:
# html = await fetch_page_on_load("https://www.naukri.com")


In [13]:
html = await fetch_page_on_load(url)
html = html.replace('\n', '').replace('\\', '')
html

Navigating to https://www.naukri.com/ai-ml-engineer-deloitte-accenture-jobs-in-hyderabad?experience=5&jobAge=15...


'<!DOCTYPE html><html lang="en"><head><meta charset="utf-8"><link rel="dns-prefetch" href="https://www.googletagmanager.com"><link rel="stylesheet" href="https://static.naukimg.com/s/9/121/_next/static/css/48b34ec3db9f7ab2.css" data-precedence="next"><link rel="stylesheet" href="https://static.naukimg.com/s/9/121/_next/static/css/78d41feaba858687.css" data-precedence="next"><link rel="stylesheet" href="https://static.naukimg.com/s/9/121/_next/static/css/e81cd227bd89f3bd.css" data-precedence="next"><link rel="stylesheet" href="https://static.naukimg.com/s/9/121/_next/static/css/7d9a0cf29dbbd70f.css" data-precedence="next"><link rel="stylesheet" href="https://static.naukimg.com/s/9/121/_next/static/css/b09725aaa4a54f31.css" data-precedence="next"><link rel="stylesheet" href="https://static.naukimg.com/s/9/121/_next/static/css/ddb0e4dfe6602a68.css" data-precedence="next"><link rel="stylesheet" href="https://static.naukimg.com/s/9/121/_next/static/css/8261187dc13c11af.css" data-precedence=

In [14]:
from bs4 import BeautifulSoup as BS

sp = BS(html, 'html.parser')
list_container = sp.find(id='listContainer')
list_container

<div class="styles_middle-section-container__iteRZ" id="listContainer"><span class="styles_failover-shimmer__0wkQi"></span><div class="styles_job-list-header-container__I1rGR" id="jobs-list-header"><div class="styles_h1-wrapper__mHVA1"><span class="styles_count-string__DlPaZ" title="1 - 20 of 861 ">1 - 20 of 861 </span><h1 class="styles_h1-content__MdLE_" title="Ai Ml Engineer Deloitte Accenture Jobs In Hyderabad Secunderabad">Ai Ml Engineer Deloitte Accenture Jobs In Hyderabad Secunderabad</h1></div><div class="styles_jlh__sort-cont__xtRva"><span class="styles_sort-by-container__ATT7x"><span class="styles_sort-by-text__SRARK">Sort by<span class="styles_blackTxt__r_Q0f">: </span></span><div class="styles_single-select-wrapper__WkB22"><button class="styles_ss__menu-btn__4s9fF styles_sort-droop-label__TxC3K" id="filter-sort" title="Relevance" type="text"><span>Relevance</span><i class="ni-icon-arrow-down"></i></button><ul class="styles_ss__menu__9TuCu styles_sort-droop-list__BmFFW" data-

In [15]:
import json
# Find all job card wrappers
job_wrappers = sp.find_all('div', class_='srp-jobtuple-wrapper')

jobs_list = []

for job in job_wrappers:
    # 0. Job ID from the wrapper attribute
    job_id = job.get('data-job-id')
    
    # 1. Job Title & URL
    title_tag = job.find('a', class_='title')
    title = title_tag.get_text(strip=True) if title_tag else None
    job_url = title_tag.get('href') if title_tag else None
    
    # 2. Company Name
    company_tag = job.find('a', class_='comp-name')
    company_name = company_tag.get_text(strip=True) if company_tag else None
    
    # 3. Company Rating
    rating_tag = job.find('span', class_='main-2')
    rating = rating_tag.get_text(strip=True) if rating_tag else None
    
    # 4. Experience Required
    exp_tag = job.find('span', class_='expwdth')
    experience = exp_tag.get('title').strip() if exp_tag and exp_tag.has_attr('title') else None
    
    # 5. Location
    loc_tag = job.find('span', class_='locWdth')
    location = loc_tag.get('title').strip() if loc_tag and loc_tag.has_attr('title') else None
    
    # 6. Job Description Summary
    desc_tag = job.find('span', class_='job-desc')
    job_description = desc_tag.get_text(strip=True) if desc_tag else None
    
    # 7. Skills / Tags
    tags_list = []
    tags_container = job.find('ul', class_='tags-gt')
    if tags_container:
        tag_items = tags_container.find_all('li', class_='tag-li')
        tags_list = [tag.get_text(strip=True) for tag in tag_items]
        
    # 8. Job Post Day / Age
    post_day_tag = job.find('span', class_='job-post-day')
    post_day = post_day_tag.get_text(strip=True) if post_day_tag else None
    
    # Construct the updated job dictionary object
    job_object = {
        "job_id": job_id,
        "title": title,
        "url": job_url,
        "company": company_name,
        "rating": rating,
        "experience": experience,
        "location": location,
        "description": job_description,
        "tags": tags_list,
        "posted_ago": post_day
    }
    
    jobs_list.append(job_object)

# Print out the extracted job objects
print(json.dumps(jobs_list, indent=4))

[
    {
        "job_id": "140926916861",
        "title": "AI / ML Engineer",
        "url": "https://www.naukri.com/job-listings-ai-ml-engineer-accenture-solutions-pvt-ltd-hyderabad-5-to-10-years-140926916861",
        "company": "Accenture",
        "rating": "3.7",
        "experience": "5-10 Yrs",
        "location": "Hyderabad",
        "description": "Minimum 5 year(s) of experience is required. Educational Qualification : 15 years full ...",
        "tags": [
            "Ml Engineer",
            "Generative Ai",
            "Artificial Intelligence",
            "Natural Language Processing",
            "Ai",
            "Neural Networks",
            "Chatbot",
            "Cloud Ai"
        ],
        "posted_ago": "1 day ago"
    },
    {
        "job_id": "140926916731",
        "title": "AI / ML Engineer",
        "url": "https://www.naukri.com/job-listings-ai-ml-engineer-accenture-solutions-pvt-ltd-hyderabad-3-to-8-years-140926916731",
        "company": "Accenture",
 

In [17]:
from sqlalchemy import JSON
from sqlalchemy import create_engine, Column, String, Integer, DateTime, Float
from sqlalchemy.orm import declarative_base, sessionmaker

Base = declarative_base()
engine = create_engine('sqlite:///test.db')

class JobDescriptionNaukri(Base):
    __tablename__='job_desc_naukri'
    id = Column(String, primary_key=True)
    job_id = Column(String)
    url = Column(String)
    company_name = Column(String)
    rating = Column(Float)
    experience = Column(String)
    location = Column(String)
    description = Column(String)
    tags = Column(JSON)
    posted_ago = Column(String)
Base.metadata.create_all(bind=engine)

Session = sessionmaker(bind=engine)


In [18]:
import uuid

jobs_list_objs = [
    JobDescriptionNaukri(
        id=str(uuid.uuid4()),
        job_id=job.get('job_id'),
        url=job.get('url'),
        company_name=job.get('company'),
        rating=float(job['rating'])
        if job.get('rating') and job['rating'].replace('.', '', 1).isdigit()
        else None,
        experience=job.get('experience'),
        location=job.get('location'),
        description=job.get('description'),
        tags=job.get('tags'),
        posted_ago=job.get('posted_ago'),
    )
    for job in jobs_list
]

db = Session()

db.add_all(jobs_list_objs)
db.commit()
db.close()

In [19]:
url = 'https://www.naukri.com/job-listings-ai-ml-engineer-accenture-solutions-pvt-ltd-hyderabad-5-to-10-years-100926933288'
html = await fetch_page_on_load(url)

Navigating to https://www.naukri.com/job-listings-ai-ml-engineer-accenture-solutions-pvt-ltd-hyderabad-5-to-10-years-100926933288...


In [21]:
import re
import json
from bs4 import BeautifulSoup

def parse_naukri_job_detail(html_content: str) -> dict:
    soup = BeautifulSoup(html_content, "html.parser")
    
    # 1. Job Title
    title_el = (
        soup.find("h1", class_=lambda c: c and "jd-header-title" in c)
        or soup.find("h1")
    )
    title = title_el.get_text(strip=True) if title_el else None

    # 2. Company Name
    comp_el = (
        soup.find("div", class_=lambda c: c and "jd-header-comp-name" in c)
        or soup.find("a", class_=lambda c: c and "jd-header-comp-name" in c)
    )
    company = comp_el.get_text(strip=True) if comp_el else None

    # 3. Experience Required
    exp_el = soup.find("div", class_=lambda c: c and "exp" in c)
    experience = exp_el.get_text(strip=True) if exp_el else None

    # 4. Salary
    sal_el = soup.find("div", class_=lambda c: c and "salary" in c)
    salary = sal_el.get_text(strip=True) if sal_el else None

    # 5. Location
    loc_el = soup.find("div", class_=lambda c: c and "location" in c)
    location = loc_el.get_text(strip=True) if loc_el else None

    # 6. Full Job Description
    # Naukri houses the main job description inside containers matching 'job-desc-container' or 'danger-html'
    jd_container = (
        soup.find("section", class_=lambda c: c and "job-desc-container" in c)
        or soup.find("div", class_=lambda c: c and "danger-html" in c)
        or soup.find("div", class_=lambda c: c and "JDC" in c)
    )
    
    job_description = ""
    if jd_container:
        # Preserves line breaks for clean reading
        job_description = jd_container.get_text(separator="\n", strip=True)

    # 7. Key Skills / Tags
    skills = []
    skill_nodes = soup.find_all("a", class_=lambda c: c and ("chip" in c or "key-skill" in c))
    for node in skill_nodes:
        skill_text = node.get_text(strip=True)
        if skill_text and skill_text not in skills:
            skills.append(skill_text)

    return {
        "title": title,
        "company": company,
        "experience": experience,
        "salary": salary,
        "location": location,
        "job_description": job_description,
        "key_skills": skills
    }

# Example Usage:
# details = parse_naukri_job_detail(html_content)
# print("Title:", details["title"])
# print("Job Description:\n", details["job_description"])


In [22]:
parse_naukri_job_detail(html)

{'title': 'AI / ML Engineer',
 'company': 'Accenture3.778.2K Reviews',
 'experience': '',
 'salary': '5 - 10 yearsNot Disclosed',
 'location': '',
 'job_description': 'Job description\nProject Role :\nAI / ML Engineer\nProject Role Description :\nDevelops applications and systems that utilize AI tools, Cloud AI services, with proper cloud or on-prem application pipeline with production ready quality. Be able to apply GenAI models as part of the solution. Could also include but not limited to deep learning, neural networks, chatbots, image processing.\nMust have skills :\nLarge Language Models (LLMs)\nGood to have skills :\nNA\nMinimum\n3\nyear(s) of experience is required\nEducational Qualification :\n15 years full time education\nSummary\n:As an AI / ML Engineer, you will engage in the development of applications and systems that leverage artificial intelligence tools and cloud AI services. Your typical day will involve designing and implementing production-ready solutions, ensuring t